# nb29 — B ~ U(1,2): CNC vs CNC+PS vs PS-only (unrotated z<0.35, B=1.35 selection mock)

Mock observables from the unrotated lightcone (z = 0–0.35) with **B=1.35** CNC selection:

| Run | Chain |
|-----|-------|
| CNC only | `chains/cnc_cosmo_arnaudB135_Y500c_unrot_z0p35_uniformB/cnc_cosmo` |
| CNC + full-sky PS | `chains/cnc_yy_combined_unrot_z0p35_arnaudB135_Y500c_uniformB/combined` |
| tSZ PS only | `chains/yy_unrot_z0p35_arnaudB135_Y500c_uniformB/yy` |

**Sampled:** `omega_cdm`, `sigma8`, `h`, `n_s`, `alpha_SZ ~ U(0.9,1.2)`, `B ~ U(1,2)`, `sigma_lnY ~ N(0.173,0.023)`

**Fixed:** `A_SZ`, `omega_b` (D3A), `tau_reio`, `m_nu`

CNC: N(q>5)=653 in 7 z-bins (0.005–0.35). PS: unrotated full-sky bandpowers + matching theory cov.

In [1]:
import os
import numpy as np
import matplotlib.pyplot as plt

# Publication-quality plot defaults
plt.rcParams.update({
    "text.usetex": False,
    "font.family": "serif",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "legend.fontsize": 10,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "axes.linewidth": 0.8,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.top": True,
    "ytick.right": True,
})

from getdist import loadMCSamples, plots

REPO = "/scratch/scratch-lxu/flamingo_repo"
OUTDIR = os.path.join(REPO, "figures", "nb29_unrot_z0p35_uniformB_contours")
os.makedirs(OUTDIR, exist_ok=True)

N_QGT5 = 653
OM_TRUTH, S8_TRUTH = 0.306, 0.808
A_SZ_FIXED = -4.28739604
LOAD_SETTINGS = {"ignore_rows": 0.1}
R1_TARGET = 0.05

RUNS = [
    dict(
        tag="cnc",
        path=os.path.join(REPO, "chains", "cnc_cosmo_arnaudB135_Y500c_unrot_z0p35_uniformB", "cnc_cosmo"),
        label="CNC only",
        color="#d62728",
        run_hint="bash cobaya/run_cnc_cosmo_B135_unrot_z0p35_uniformB.sh",
    ),
    dict(
        tag="comb",
        path=os.path.join(REPO, "chains", "cnc_yy_combined_unrot_z0p35_arnaudB135_Y500c_uniformB", "combined"),
        label="CNC + full-sky PS",
        color="#8c564b",
        run_hint="bash cobaya/run_cnc_yy_combined_unrot_z0p35_B135_uniformB.sh",
    ),
    dict(
        tag="yy",
        path=os.path.join(REPO, "chains", "yy_unrot_z0p35_arnaudB135_Y500c_uniformB", "yy"),
        label="tSZ PS only",
        color="#2ca02c",
        run_hint="bash cobaya/run_yy_unrot_z0p35_B135_uniformB.sh",
    ),
]

TRUTH = {
    "Omega_m": OM_TRUTH, "sigma8": S8_TRUTH,
    "S8": S8_TRUTH * np.sqrt(OM_TRUTH / 0.3),
    "F": S8_TRUTH * (OM_TRUTH / 1.1) ** 0.40 * 0.681 ** (-0.21),
    "h": 0.681, "omega_b": 0.022539, "n_s": 0.967,
    "alpha_SZ": 1.12, "sigma_lnY": 0.173, "B": 1.35,
}
CHAIN_PARS = ["Omega_m", "sigma8", "h", "n_s", "alpha_SZ", "B", "sigma_lnY"]
DERIVED_PARS = ["S8", "F"]
COSMO_PARS = ["Omega_m", "sigma8", "S8", "F"]
PARS_PRINT = CHAIN_PARS[:2] + DERIVED_PARS + CHAIN_PARS[2:]
FIXED_PARS = {"A_SZ": A_SZ_FIXED, "omega_b": 0.022539}


def _add_derived(sample):
    p = sample.getParams()
    sample.addDerived(
        p.sigma8 * np.sqrt(p.Omega_m / 0.3), name="S8", label=r"S_8",
    )
    h = getattr(p, "h", 0.681)  # CCCP mean when h fixed in chain
    sample.addDerived(
        p.sigma8 * (p.Omega_m / p.B) ** 0.40 * h ** (-0.21), name="F", label=r"F",
    )
    return sample


def _load_chain(path, label, run_hint):
    if not os.path.exists(path + ".1.txt"):
        print(f"MISSING: {path}")
        print(f"  run: {run_hint}")
        return None
    sample = loadMCSamples(path, settings=LOAD_SETTINGS)
    sample = _add_derived(sample)
    r1 = getattr(sample, "Rminus1", None)
    print(f"\n{label}: {sample.numrows} post-discard samples", end="")
    if r1 is not None:
        flag = " (converged)" if r1 <= R1_TARGET else " (running)"
        print(f"; R-1={r1:.3f}{flag}", end="")
    print()
    names = set(sample.getParamNames().list())
    print(f"{label} — unrot z<0.35 B135 mock (N={N_QGT5}):")
    for nm in PARS_PRINT:
        if nm in names:
            print(f"  {nm:12s} = {sample.mean(nm):.4f} +/- {sample.std(nm):.4f}")
    for nm, val in FIXED_PARS.items():
        print(f"  {nm:12s} = {val:.6f} (fixed)")
    return sample


loaded = {}
for run in RUNS:
    s = _load_chain(run["path"], run["label"], run["run_hint"])
    if s is not None:
        loaded[run["tag"]] = s

if "cnc" in loaded and "comb" in loaded:
    s_cnc, s_comb = loaded["cnc"], loaded["comb"]
    print(f"\nΔS8 (comb − CNC) = {s_comb.mean('S8') - s_cnc.mean('S8'):+.4f}")
    print(f"ΔB  (comb − CNC) = {s_comb.mean('B') - s_cnc.mean('B'):+.4f}")

chains = [loaded[r["tag"]] for r in RUNS if r["tag"] in loaded]
labels = [r["label"] for r in RUNS if r["tag"] in loaded]
colors = [r["color"] for r in RUNS if r["tag"] in loaded]

if len(chains) < 2:
    print("\nNeed at least two chains for contour plots — launch Cobaya runs first.")
else:
    names = [set(s.getParamNames().list()) for s in chains]
    full_pars = CHAIN_PARS[:2] + DERIVED_PARS + CHAIN_PARS[2:]
    plot_pars = [p for p in full_pars if all(p in n for n in names)]
    cosmo_pars = [p for p in COSMO_PARS if all(p in n for n in names)]



CNC only: 5328 post-discard samples
CNC only — unrot z<0.35 B135 mock (N=653):
  Omega_m      = 0.3019 +/- 0.0362
  sigma8       = 0.7739 +/- 0.0448
  S8           = 0.7741 +/- 0.0535
  F            = 0.4451 +/- 0.0051
  h            = 0.6822 +/- 0.0230
  n_s          = 0.9663 +/- 0.0135
  alpha_SZ     = 1.0563 +/- 0.0782
  B            = 1.4793 +/- 0.2660
  sigma_lnY    = 0.1758 +/- 0.0231
  A_SZ         = -4.287396 (fixed)
  omega_b      = 0.022539 (fixed)

CNC + full-sky PS: 7236 post-discard samples
CNC + full-sky PS — unrot z<0.35 B135 mock (N=653):
  Omega_m      = 0.2995 +/- 0.0091
  sigma8       = 0.7166 +/- 0.0108
  S8           = 0.7157 +/- 0.0043
  F            = 0.4774 +/- 0.0027
  h            = 0.6915 +/- 0.0254
  n_s          = 0.9773 +/- 0.0126
  alpha_SZ     = 0.9067 +/- 0.0070
  B            = 1.0030 +/- 0.0029
  sigma_lnY    = 0.2131 +/- 0.0213
  A_SZ         = -4.287396 (fixed)
  omega_b      = 0.022539 (fixed)

tSZ PS only: 10962 post-discard samples
tSZ PS only —

## Triangle: cosmology + S8, F + nuisances (incl. B, α, σ_lnY)

In [2]:
if len(chains) >= 2 and plot_pars:  # CNC, combined, and/or PS-only
    g = plots.get_subplot_plotter(width_inch=8)
    g.settings.legend_fontsize = 10
    g.settings.figure_legend_frame = False
    g.triangle_plot(
        chains, plot_pars, filled=True, contour_colors=colors,
        legend_labels=labels, legend_loc="upper center", legend_ncol=3,
        title_limit=1, markers={k: TRUTH[k] for k in plot_pars if k in TRUTH},
    )
    g.fig.suptitle(
        r"$B \sim U(1,2)$: unrotated $z<0.35$ (CNC / combined / PS-only)",
        fontsize=12,
    )
    g.fig.subplots_adjust(top=0.90)
    g.export(os.path.join(OUTDIR, "unrot_z0p35_uniformB_triangle.pdf"))
    g.export(os.path.join(OUTDIR, "unrot_z0p35_uniformB_triangle.png"), dpi=300)
    plt.show()
else:
    print("Triangle skipped.")


## Ω_m–σ_8 plane

In [3]:
from matplotlib.lines import Line2D

if len(chains) >= 2:
    g2 = plots.get_subplot_plotter(width_inch=5)
    g2.plot_2d(chains, "Omega_m", "sigma8", filled=True, colors=colors)
    ax = plt.gca()
    ax.plot(OM_TRUTH, S8_TRUTH, "*", color="crimson", ms=12, mec="k", zorder=10)
    ax.axvline(OM_TRUTH, color="k", ls="--", lw=1, alpha=0.6)
    ax.axhline(S8_TRUTH, color="k", ls="--", lw=1, alpha=0.6)
    handles = [
        Line2D([], [], color=c, lw=8, alpha=0.7, label=l)
        for c, l in zip(colors, labels)
    ] + [
        Line2D([], [], color="crimson", marker="*", ls="", ms=12, mec="k", label="FLAMINGO truth"),
    ]
    ax.legend(handles=handles, fontsize=9, loc="upper left", frameon=False)
    ax.set_title(r"Unrot $z<0.35$: CNC vs CNC+full-sky PS", fontsize=11)
    g2.export(os.path.join(OUTDIR, "unrot_z0p35_uniformB_Om_s8_plane.pdf"))
    g2.export(os.path.join(OUTDIR, "unrot_z0p35_uniformB_Om_s8_plane.png"), dpi=300)
    print("saved ->", OUTDIR)
    plt.show()
else:
    print("Ωm–σ8 plane skipped.")


saved -> /scratch/scratch-lxu/flamingo_repo/figures/nb29_unrot_z0p35_uniformB_contours


---

## CCCP prior on $1{-}b$ (same unrot z<0.35 mock)

Informative bias prior from CCCP: **`one_minus_b ~ N(0.780, 0.092)`**, **`B = 1/(1-b)`** (avoids flat-$B$ prior-volume artefact in combined runs above).

| Run | Chain |
|-----|-------|
| CNC only | `chains/cnc_cosmo_arnaudB135_Y500c_unrot_z0p35_cccp/cnc_cosmo` |
| CNC + full-sky PS | `chains/cnc_yy_combined_unrot_z0p35_arnaudB135_Y500c_cccp/combined` |
| tSZ PS only | `chains/yy_unrot_z0p35_arnaudB135_Y500c_cccp/yy` |

In [4]:
CCCP_RUNS = [
    dict(tag="cnc", path=os.path.join(REPO, "chains", "cnc_cosmo_arnaudB135_Y500c_unrot_z0p35_cccp", "cnc_cosmo"),
         label="CNC only (CCCP)", color="#d62728",
         run_hint="bash cobaya/run_cnc_cosmo_B135_unrot_z0p35_cccp.sh"),
    dict(tag="comb", path=os.path.join(REPO, "chains", "cnc_yy_combined_unrot_z0p35_arnaudB135_Y500c_cccp", "combined"),
         label="CNC + PS (CCCP)", color="#8c564b",
         run_hint="bash cobaya/run_cnc_yy_combined_unrot_z0p35_B135_cccp.sh"),
    dict(tag="yy", path=os.path.join(REPO, "chains", "yy_unrot_z0p35_arnaudB135_Y500c_cccp", "yy"),
         label="PS only (CCCP)", color="#2ca02c",
         run_hint="bash cobaya/run_yy_unrot_z0p35_B135_cccp.sh"),
]

cccp_loaded = {}
for run in CCCP_RUNS:
    s = _load_chain(run["path"], run["label"], run["run_hint"])
    if s is not None:
        cccp_loaded[run["tag"]] = s

if "cnc" in cccp_loaded and "comb" in cccp_loaded:
    print(f"\n[CCCP] ΔS8 (comb − CNC) = {cccp_loaded['comb'].mean('S8') - cccp_loaded['cnc'].mean('S8'):+.4f}")
    print(f"[CCCP] ΔB  (comb − CNC) = {cccp_loaded['comb'].mean('B') - cccp_loaded['cnc'].mean('B'):+.4f}")

cccp_chains = [cccp_loaded[r["tag"]] for r in CCCP_RUNS if r["tag"] in cccp_loaded]
cccp_labels = [r["label"] for r in CCCP_RUNS if r["tag"] in cccp_loaded]
cccp_colors = [r["color"] for r in CCCP_RUNS if r["tag"] in cccp_loaded]

CCCP_OUTDIR = os.path.join(REPO, "figures", "nb29_unrot_z0p35_cccp_contours")
os.makedirs(CCCP_OUTDIR, exist_ok=True)


CNC only (CCCP): 2880 post-discard samples
CNC only (CCCP) — unrot z<0.35 B135 mock (N=653):
  Omega_m      = 0.3090 +/- 0.0222
  sigma8       = 0.7337 +/- 0.0228
  S8           = 0.7438 +/- 0.0290
  F            = 0.4474 +/- 0.0041
  h            = 0.6823 +/- 0.0220
  n_s          = 0.9664 +/- 0.0141
  alpha_SZ     = 1.1105 +/- 0.0444
  B            = 1.3029 +/- 0.1369
  sigma_lnY    = 0.1739 +/- 0.0238
  A_SZ         = -4.287396 (fixed)
  omega_b      = 0.022539 (fixed)
/scratch/scratch-lxu/flamingo_repo/chains/cnc_yy_combined_unrot_z0p35_arnaudB135_Y500c_cccp/combined.1.txt
Removed 0.1 as burn in

CNC + PS (CCCP): 6282 post-discard samples
CNC + PS (CCCP) — unrot z<0.35 B135 mock (N=653):
  Omega_m      = 0.2111 +/- 0.0054
  sigma8       = 0.7045 +/- 0.0041
  S8           = 0.5908 +/- 0.0072
  F            = 0.4934 +/- 0.0027
  h            = 0.7065 +/- 0.0182
  n_s          = 0.9781 +/- 0.0134
  alpha_SZ     = 0.8268 +/- 0.0340
  B            = 0.6171 +/- 0.0196
  sigma_lnY    = 0

In [5]:
if len(cccp_chains) >= 2:
    g_cccp = plots.getSubplotPlotter()
    g_cccp.settings.legend_fontsize = 9
    g_cccp.settings.figure_legend_frame = False
    g_cccp.triangle_plot(
        cccp_chains, params=plot_pars, legend_labels=cccp_labels,
        colors=cccp_colors, filled=True,
        legend_loc="upper center", legend_ncol=1,
        markers={k: TRUTH[k] for k in plot_pars if k in TRUTH},
        marker_color="k", marker_style="*", marker_size=80,
    )
    g_cccp.fig.suptitle(
        r"CCCP $1{-}b$ prior: unrot z$\leq$0.35, B=1.35 CNC sel.", fontsize=11, y=1.02,
    )
    g_cccp.export(os.path.join(CCCP_OUTDIR, "unrot_z0p35_cccp_triangle.pdf"))
    g_cccp.export(os.path.join(CCCP_OUTDIR, "unrot_z0p35_cccp_triangle.png"), dpi=300)
    plt.show()
    print("saved ->", CCCP_OUTDIR)
else:
    print("CCCP chains not ready — launch cobaya/run_*_cccp.sh scripts.")


saved -> /scratch/scratch-lxu/flamingo_repo/figures/nb29_unrot_z0p35_cccp_contours


---

## CCCP + fixed $h$, $n_s$, $\alpha_{\rm SZ}$

Same CCCP setup but **`h=0.681`, `n_s=0.967`, `alpha_SZ=1.12` fixed**; at CCCP means; other priors unchanged.
Chains: `bash cobaya/run_*_cccp_fixAlpha112.sh` (full-sky or unrot as above).


In [6]:
FIXALPHA_RUNS = [
    dict(tag="cnc", path=os.path.join(REPO, "chains", "cnc_cosmo_arnaudB135_Y500c_unrot_z0p35_cccp_fixAlpha112", "cnc_cosmo"),
         label="CNC only (fix α)", color="#d62728",
         run_hint="bash cobaya/run_cnc_cosmo_B135_unrot_z0p35_cccp_fixAlpha112.sh"),
    dict(tag="comb", path=os.path.join(REPO, "chains", "cnc_yy_combined_unrot_z0p35_arnaudB135_Y500c_cccp_fixAlpha112", "combined"),
         label="CNC + PS (fix α)", color="#8c564b",
         run_hint="bash cobaya/run_cnc_yy_combined_unrot_z0p35_B135_cccp_fixAlpha112.sh"),
    dict(tag="yy", path=os.path.join(REPO, "chains", "yy_unrot_z0p35_arnaudB135_Y500c_cccp_fixAlpha112", "yy"),
         label="PS only (fix α)", color="#2ca02c",
         run_hint="bash cobaya/run_yy_unrot_z0p35_B135_cccp_fixAlpha112.sh"),
]

fixalpha_loaded = {}
for run in FIXALPHA_RUNS:
    s = _load_chain(run["path"], run["label"], run["run_hint"])
    if s is not None:
        fixalpha_loaded[run["tag"]] = s

if "cnc" in fixalpha_loaded and "comb" in fixalpha_loaded:
    print(f"\n[fix α] ΔS8 (comb − CNC) = {fixalpha_loaded['comb'].mean('S8') - fixalpha_loaded['cnc'].mean('S8'):+.4f}")
    print(f"[fix α] ΔB  (comb − CNC) = {fixalpha_loaded['comb'].mean('B') - fixalpha_loaded['cnc'].mean('B'):+.4f}")

fixalpha_chains = [fixalpha_loaded[r["tag"]] for r in FIXALPHA_RUNS if r["tag"] in fixalpha_loaded]
fixalpha_labels = [r["label"] for r in FIXALPHA_RUNS if r["tag"] in fixalpha_loaded]
fixalpha_colors = [r["color"] for r in FIXALPHA_RUNS if r["tag"] in fixalpha_loaded]

FIXALPHA_OUTDIR = os.path.join(REPO, "figures", "nb29_unrot_z0p35_cccp_fixAlpha112_contours")
os.makedirs(FIXALPHA_OUTDIR, exist_ok=True)
fixalpha_plot_pars = [p for p in plot_pars if p not in ("alpha_SZ", "h", "n_s")]


/scratch/scratch-lxu/flamingo_repo/chains/cnc_cosmo_arnaudB135_Y500c_unrot_z0p35_cccp_fixAlpha112/cnc_cosmo.1.txt
Removed 0.1 as burn in

CNC only (fix α): 1368 post-discard samples
CNC only (fix α) — unrot z<0.35 B135 mock (N=653):
  Omega_m      = 0.3161 +/- 0.0152
  sigma8       = 0.7304 +/- 0.0184
  S8           = 0.7496 +/- 0.0244
  F            = 0.4465 +/- 0.0032
  B            = 1.3263 +/- 0.1168
  sigma_lnY    = 0.1752 +/- 0.0219
  A_SZ         = -4.287396 (fixed)
  omega_b      = 0.022539 (fixed)
/scratch/scratch-lxu/flamingo_repo/chains/cnc_yy_combined_unrot_z0p35_arnaudB135_Y500c_cccp_fixAlpha112/combined.1.txt
Removed 0.1 as burn in

CNC + PS (fix α): 2538 post-discard samples
CNC + PS (fix α) — unrot z<0.35 B135 mock (N=653):
  Omega_m      = 0.2151 +/- 0.0055
  sigma8       = 0.7032 +/- 0.0031
  S8           = 0.5954 +/- 0.0075
  F            = 0.4988 +/- 0.0019
  B            = 0.6211 +/- 0.0194
  sigma_lnY    = 0.1999 +/- 0.0249
  A_SZ         = -4.287396 (fixed)
  ome

In [7]:
if len(fixalpha_chains) >= 2:
    g_fa = plots.getSubplotPlotter()
    g_fa.settings.legend_fontsize = 9
    g_fa.settings.figure_legend_frame = False
    g_fa.triangle_plot(
        fixalpha_chains, params=fixalpha_plot_pars, legend_labels=fixalpha_labels,
        colors=fixalpha_colors, filled=True,
        legend_loc="upper center", legend_ncol=1,
        markers={k: TRUTH[k] for k in fixalpha_plot_pars if k in TRUTH},
        marker_color="k", marker_style="*", marker_size=80,
    )
    g_fa.fig.suptitle(r"CCCP, fixed h, n_s, α: unrot z\\leq0.35, B=1.35 CNC sel.", fontsize=11, y=1.02)
    g_fa.export(os.path.join(FIXALPHA_OUTDIR, "unrot_z0p35_cccp_fixAlpha112_triangle.pdf"))
    g_fa.export(os.path.join(FIXALPHA_OUTDIR, "unrot_z0p35_cccp_fixAlpha112_triangle.png"), dpi=300)
    plt.show()
    print("saved ->", FIXALPHA_OUTDIR)
else:
    print("fixAlpha chains not ready yet.")


saved -> /scratch/scratch-lxu/flamingo_repo/figures/nb29_unrot_z0p35_cccp_fixAlpha112_contours
